# Cohort Analysis & Customer Segmentation

This project analyzes customer retention and purchasing behavior using **Cohort Analysis** and **RFM segmentation**.

### Business questions
- How many customers return after their first purchase?
- How does retention change across customer cohorts?
- Which cohorts generate the most revenue?
- Which customer segments are the most valuable?

> **Note:** Profit and churn prediction are intentionally excluded. Profit requires a real margin assumption, while churn prediction should use a future observation window to avoid target leakage. This notebook focuses on metrics that can be directly supported by the transaction data.

## 1. Setup

The notebook uses a relative data path so it can run on another machine after cloning the repository.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

# Project paths
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

DATA_FILE = DATA_DIR / "online_retail_II.xlsx"

print(f"Data file: {DATA_FILE}")
print(f"Exists: {DATA_FILE.exists()}")

## 2. Load and inspect the data

In [ ]:
df = pd.read_excel(DATA_FILE)

print("Shape:", df.shape)
display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

## 3. Data cleaning

The cleaning rules below are based on transaction validity: missing customer IDs cannot be used for customer-level analysis, negative quantities represent returns/cancellations, and non-product service rows are excluded from product analysis.

In [ ]:
df = df.copy()

# Standardize column names used in the analysis
df = df.rename(columns={"Customer ID": "CustomerID", "Price": "UnitPrice"})

# Convert dates and remove rows without the fields required for customer analysis
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
required_cols = ["CustomerID", "InvoiceDate", "Invoice", "Quantity", "UnitPrice"]
df = df.dropna(subset=required_cols).copy()

# Remove duplicate transaction rows
df = df.drop_duplicates().copy()

# Keep completed sales transactions
df = df[df["Quantity"] > 0].copy()

# Exclude service / non-product rows from product-level analysis
special_items = {"Manual", "Discount", "POSTAGE", "CRUK Commission", "DOTCOM POSTAGE"}
if "Description" in df.columns:
    df = df[~df["Description"].isin(special_items)].copy()

# Keep plausible stock codes when the column exists
if "StockCode" in df.columns:
    df = df[df["StockCode"].astype(str).str.len() >= 5].copy()

# Revenue per line
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df = df.reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Unique customers:", df["CustomerID"].nunique())
print("Total revenue:", f"{df['Revenue'].sum():,.2f}")

## 4. Cohort assignment

A customer's cohort is the month of their **first recorded purchase**. The cohort index is calculated as the difference between calendar months, rather than dividing days by 30. This avoids errors around month lengths.

In [ ]:
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")

first_purchase = (
    df.groupby("CustomerID")["InvoiceMonth"]
      .min()
      .rename("CohortMonth")
)

df = df.join(first_purchase, on="CustomerID")

# Exact calendar-month difference
df["CohortIndex"] = (
    (df["InvoiceMonth"].dt.year - df["CohortMonth"].dt.year) * 12
    + (df["InvoiceMonth"].dt.month - df["CohortMonth"].dt.month)
)

print(df[["CustomerID", "InvoiceMonth", "CohortMonth", "CohortIndex"]].head(10))

## 5. Customer retention

Retention is measured as the percentage of customers from each cohort who purchased again in a given month index.

In [ ]:
cohort_counts = (
    df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"]
      .nunique()
      .reset_index(name="CustomerCount")
)

cohort_pivot = cohort_counts.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="CustomerCount"
)

cohort_size = cohort_pivot[0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100

print("Retention rate (%):")
display(retention.round(1).head(10))

In [ ]:
plt.figure(figsize=(16, 9))
sns.heatmap(
    retention,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Retention (%)"}
)
plt.title("Customer Retention by Cohort")
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort Month")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cohort_retention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Revenue by cohort

Revenue is aggregated by acquisition cohort and months since first purchase. This complements retention because a cohort with fewer customers can still generate substantial revenue.

In [ ]:
revenue_cohort = (
    df.groupby(["CohortMonth", "CohortIndex"])["Revenue"]
      .sum()
      .reset_index()
)

revenue_pivot = revenue_cohort.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="Revenue"
)

display(revenue_pivot.round(2).head(10))

plt.figure(figsize=(16, 9))
sns.heatmap(
    revenue_pivot,
    annot=True,
    fmt=".0f",
    cmap="OrRd",
    cbar_kws={"label": "Revenue"}
)
plt.title("Revenue by Customer Cohort")
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort Month")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cohort_revenue_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. RFM analysis

RFM summarizes each customer's behavior: **Recency** (days since last purchase), **Frequency** (number of invoices), and **Monetary** (total revenue). The reference date is one day after the latest transaction.

In [ ]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby("CustomerID")
      .agg(
          Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
          Frequency=("Invoice", "nunique"),
          Monetary=("Revenue", "sum")
      )
      .reset_index()
)

# Quartile-based scores: higher is better for all three dimensions
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]
).astype(int)
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]
).astype(int)
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]
).astype(int)

rfm["RFM_Score"] = rfm[["R_Score", "F_Score", "M_Score"]].sum(axis=1)

# Simple, interpretable customer segments
def assign_segment(score):
    if score >= 10:
        return "Champions"
    if score >= 8:
        return "Loyal Customers"
    if score >= 6:
        return "Potential Loyalists"
    if score >= 4:
        return "At Risk"
    return "Needs Attention"

rfm["RFM_Segment"] = rfm["RFM_Score"].apply(assign_segment)

print("Customers:", len(rfm))
display(rfm.head())
display(rfm["RFM_Segment"].value_counts().rename_axis("Segment").to_frame("Customers"))

In [ ]:
segment_summary = (
    rfm.groupby("RFM_Segment")
       .agg(
           Customers=("CustomerID", "count"),
           AvgRecency=("Recency", "mean"),
           AvgFrequency=("Frequency", "mean"),
           AvgMonetary=("Monetary", "mean"),
           TotalRevenue=("Monetary", "sum")
       )
       .sort_values("TotalRevenue", ascending=False)
       .round(2)
)

display(segment_summary)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=segment_summary.reset_index(),
    x="RFM_Segment",
    y="TotalRevenue"
)
plt.title("Revenue by RFM Segment")
plt.xlabel("RFM Segment")
plt.ylabel("Total Revenue")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rfm_segment_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Customer clustering

K-Means is used as an exploratory segmentation method on standardized RFM variables. The elbow curve is inspected before selecting a practical value of K. Cluster labels are arbitrary, so clusters are interpreted using their RFM averages rather than their numeric IDs.

In [ ]:
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

# Log transform reduces the influence of highly skewed customer values
rfm_log = np.log1p(rfm_features)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

wcss = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(n_clusters=k, n_init=20, random_state=42)
    model.fit(rfm_scaled)
    wcss.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.xticks(list(k_values))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "elbow_method.png", dpi=150, bbox_inches="tight")
plt.show()

# Four clusters provide an interpretable starting point; validate this choice visually.
K = 4
kmeans = KMeans(n_clusters=K, n_init=20, random_state=42)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

cluster_summary = (
    rfm.groupby("Cluster")
       .agg(
           Customers=("CustomerID", "count"),
           AvgRecency=("Recency", "mean"),
           AvgFrequency=("Frequency", "mean"),
           AvgMonetary=("Monetary", "mean"),
           TotalRevenue=("Monetary", "sum")
       )
       .sort_values("TotalRevenue", ascending=False)
       .round(2)
)

display(cluster_summary)

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=rfm.sample(min(1000, len(rfm)), random_state=42),
    x="Recency",
    y="Monetary",
    hue="Cluster",
    palette="Set2",
    alpha=0.7
)
plt.title("Customer Clusters: Recency vs Monetary")
plt.xlabel("Recency (days)")
plt.ylabel("Monetary (revenue)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "customer_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Business takeaways

Use the tables and charts above to write the final conclusions for the actual dataset. Avoid generic claims; the statements below should be replaced with the observed numbers from the run.

Suggested interpretation framework:

- **Retention:** Identify where the largest drop occurs after month 0.
- **Cohorts:** Compare cohorts with unusually strong or weak retention.
- **Revenue:** Check whether high-revenue cohorts also have strong retention.
- **RFM:** Prioritize Champions and Loyal Customers for retention and cross-sell opportunities.
- **At Risk:** Review customers with high historical value but weak recency for reactivation campaigns.
- **Clusters:** Describe each cluster using its average Recency, Frequency, and Monetary values rather than the cluster number alone.

In [ ]:
# Export analysis outputs for the GitHub repository
rfm.to_csv(OUTPUT_DIR / "customer_rfm.csv", index=False, encoding="utf-8-sig")
retention.to_csv(OUTPUT_DIR / "cohort_retention_table.csv", encoding="utf-8-sig")
revenue_pivot.to_csv(OUTPUT_DIR / "cohort_revenue_table.csv", encoding="utf-8-sig")
cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary.csv", encoding="utf-8-sig")
segment_summary.to_csv(OUTPUT_DIR / "rfm_segment_summary.csv", encoding="utf-8-sig")

summary = {
    "customers": int(rfm["CustomerID"].nunique()),
    "transactions": int(df["Invoice"].nunique()),
    "revenue": float(df["Revenue"].sum()),
    "date_start": str(df["InvoiceDate"].min().date()),
    "date_end": str(df["InvoiceDate"].max().date()),
    "top_rfm_segment": str(segment_summary.index[0]),
}

with open(OUTPUT_DIR / "analysis_summary.txt", "w", encoding="utf-8") as f:
    for key, value in summary.items():
        f.write(f"{key}: {value}\n")

print("Analysis outputs saved to:", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)